<a href="https://colab.research.google.com/github/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution/blob/main/notebooks/06_collect_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collect results for the repository

Notebooks 01-05 save their results on Google Drive (`thesis-data/`). This notebook gathers the small summary files the thesis tables
come from into one zip with the same folder layout as the repository's `outputs/` folder, so they can be committed to GitHub.

**Included:** temperature and training history per model, CNN stress test, watermark threshold and results, provenance results and attacks,
fusion and no-watermark ablation summaries, measured watermark confidence, the near-duplicate report and the results without near-duplicates, and the train/val/test split lists.
**Left on Drive:** model checkpoints (`*.pt`, too large for git), per-image CSVs (several MB each, can be regenerated) and the images.

**No GPU needed**: Runtime > Change runtime type > **CPU**. Takes under a minute. Run it again whenever results change.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Collect the summary files

In [ ]:
import datetime, fnmatch, glob, json, os, shutil, zipfile

D = "/content/drive/MyDrive/thesis-data"   # where notebooks 01-05 save their results
OUT = "/content/thesis-results"            # staging folder; its outputs/ mirrors the repo's outputs/

# Small summary files only: no model checkpoints (*.pt) and no per-image CSVs.
# Keep in sync with the outputs/ section of .gitignore, which lets exactly these into git.
PATTERNS = [
    "models/*/calibration.json",           # notebook 02: temperature scaling
    "models/*/history.json",               # notebook 02: training curves
    "stress_test/*/results.csv",           # notebook 02: CNN under transforms, delta-A
    "stress_test/*/results.json",
    "watermark/threshold.json",            # notebook 03
    "watermark/results.csv",
    "provenance/results.csv",              # notebook 04
    "provenance/attacks.csv",
    "provenance/softbinding.json",
    "evaluation/*/summary.csv",            # notebook 05: fusion and the no-watermark ablation
    "evaluation/*/summary.json",
    "evaluation/*/watermark_confidence.json",
    "evaluation/*/summary_excl_near_duplicates.csv",   # notebook 05b: results without near-duplicates
    "evaluation/*/summary_excl_near_duplicates.json",
    "evaluation/*/near_duplicate_sensitivity.csv",
    "data_quality/near_duplicates.csv",                # notebook 01b: near-duplicate pairs across splits
    "data_quality/near_duplicates_summary.json",
    "data_quality/removed_near_duplicates.csv",        # notebook 01c: images removed before retraining
    "data_quality/removed_near_duplicates_summary.json",
    "data_quality/after_removal/near_duplicates_summary.json",
    "cv/folds.csv",                                     # notebook 07: cross-validation
    "cv/summary_by_fold.csv",
    "cv/summary_mean_sd.csv",
    "cv/significance.csv",
    "cv/cv_report.json",
    "cv/fold_*/summary.csv",
    "cv/fold_*/watermark.json",
    "cv/fold_*/*/calibration.json",
    "cv/fold_*/*/history.json",
]
SPLITS = ["train.csv", "val.csv", "test.csv"]  # which images were in each split (from harmonized.zip)
MAX_MB = 10                                     # anything bigger doesn't belong in git

shutil.rmtree(OUT, ignore_errors=True)
copied, missing = [], []
for pattern in PATTERNS:
    matches = sorted(glob.glob(os.path.join(D, pattern)))
    if not matches:
        missing.append(pattern)
    for src in matches:
        rel = os.path.relpath(src, D)
        size_mb = os.path.getsize(src) / 1e6
        if size_mb > MAX_MB:
            print(f"skipped {rel}: {size_mb:.1f} MB is too large for the repository")
            continue
        dst = os.path.join(OUT, "outputs", rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        modified = datetime.datetime.fromtimestamp(os.path.getmtime(src)).isoformat(timespec="seconds")
        copied.append({"path": f"outputs/{rel}", "bytes": os.path.getsize(src), "modified": modified})

zip_path = os.path.join(D, "harmonized.zip")
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist():
            if any(fnmatch.fnmatch(name, f"*harmonized/{s}") for s in SPLITS):
                dst = os.path.join(OUT, "outputs", "splits", os.path.basename(name))
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                with z.open(name) as f, open(dst, "wb") as g:
                    shutil.copyfileobj(f, g)
                copied.append({"path": f"outputs/splits/{os.path.basename(name)}", "bytes": os.path.getsize(dst),
                               "modified": None})
else:
    missing.append("harmonized.zip (split CSVs)")

manifest = {"collected_at": datetime.datetime.now().isoformat(timespec="seconds"), "source": D,
            "files": copied, "missing": missing}
with open(os.path.join(OUT, "outputs", "MANIFEST.json"), "w") as f:
    json.dump(manifest, f, indent=1)

for c in copied:
    print(f"  {c['path']:<60} {c['bytes'] / 1e3:8.1f} kB")
print(f"\n{len(copied)} files, {sum(c['bytes'] for c in copied) / 1e6:.2f} MB in total")
if missing:
    print("not found (run that notebook first, or ignore if you skipped it):", *missing, sep="\n  ")
archive = shutil.make_archive("/content/thesis-results", "zip", OUT)
print(f"\nzipped to {archive}")

## 3. Download the zip

In [ ]:
from google.colab import files
files.download("/content/thesis-results.zip")

## 4. Add it to GitHub
Unzip `thesis-results.zip` on your computer; it contains one folder, `outputs/`. Then either:

- **On github.com:** open the repository, choose **Add file > Upload files**, drag the `outputs` folder in, and commit to a new branch
  so it can be reviewed as a pull request, or
- **With git:** copy the folder into your clone, then `git add outputs/` and commit.

`.gitignore` lets exactly these summary files into `outputs/`; model checkpoints, per-image CSVs and anything else there stay ignored.
If you add a new kind of result, add its pattern both to `PATTERNS` above and to the `outputs/` section of `.gitignore`.
`outputs/MANIFEST.json` lists every file with its size and when it was last changed on Drive, plus anything that was missing.